# ResNet-50 混合 SRAM/PCM-IMC 精度实验 (ImageNet, H100)

和 LeNet 实验同一套 IMC 模型:cellBit=2 符号-幅值 cell、二补码逐位输入、64 行子阵分块、5-bit ADC、逐权重漂移 t^(-nu)、未保护 cell 上的写噪声;保护(top-k)权重精确(SRAM)。

**臂**:none / uniform(beta=5%) / layer_aware(Table IX 分配)。每臂一个 clean 点(全 INT8,无漂移/ADC/噪声) + {无ADC, 5-bit ADC} x 漂移网格 x 多 seed 的扫描。

**前提**:你的 `PCM_Research7` 已在 Google Drive(含 `kaggle.json` 和 `outputs_ResNet50_ImageNet/LayerAware_Ratios_ResNet50.json`)。本 notebook 不需要你上传任何 .py(用 %%writefile 直接写)。

**说明**:存储漂移的全量 5 万张是 Paper 1 的;IMC+ADC 这条作为大网络上的验证,跑代表性子集。clean 点用全 INT8(权重+激活,因为 crossbar 需要数字输入),会比 76.14% 略低一点点(激活量化代价),论文里有意义的是同一实验内各臂之间的对比。


## 0. 环境检查 (确认是 H100)

In [1]:
!nvidia-smi


Thu Jun 11 14:25:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. 挂载 Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/PCM_Research7'
!ls {PROJECT}


Mounted at /content/drive
config_imagenet.py			  Step_B1_Aggregate.py
kaggle.json				  Step_B1_LayerSensitivity.py
outputs_ResNet50_ImageNet		  Step_B2_BudgetSweep.py
PCM_Research5_Data_Step3_Sensitivity.csv  Step_B3_LongTerm.py
PCM_Research6_Data_Step3_Sensitivity.csv  Step_B4_Mismatch.py
README.md				  Step_B5_Scaling.py
Run_on_Colab.ipynb			  Step_B6_PrecisionAblation.py
Step_B0_VerifyBaseline.py		  test_dry_run.py


## 2. 下载 ImageNet 验证集 (6.7 GB, 复用你原来的 Kaggle 方式)

In [3]:
# 路径 A：Kaggle 下载（和 PCM_Research7 一致）
import os, time, glob
!mkdir -p ~/.kaggle
!cp {PROJECT}/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

VAL_ROOT = '/content/imagenet_val'
need_download = (not os.path.exists(VAL_ROOT) or
                 len([d for d in os.listdir(VAL_ROOT) if d.startswith('n0')]) < 100)
if need_download:
    !mkdir -p {VAL_ROOT}
    t0 = time.time(); print('Downloading from Kaggle (titericz/imagenet1k-val) ...')
    !kaggle datasets download -d titericz/imagenet1k-val -p /content --quiet
    print(f'Downloaded in {time.time()-t0:.0f}s'); print('Unzipping (50K images) ...')
    t0 = time.time()
    !unzip -q /content/imagenet1k-val.zip -d {VAL_ROOT}
    !rm /content/imagenet1k-val.zip
    print(f'Unzipped in {time.time()-t0:.0f}s')
else:
    print('Val set already present.')

# 自动探测真实的 ImageFolder 根目录（可能有嵌套）
cand = glob.glob(f'{VAL_ROOT}/**/n01440764', recursive=True)
if cand:
    VAL_ROOT = os.path.dirname(cand[0])
    syn = [d for d in os.listdir(VAL_ROOT) if d.startswith('n0')]
    print(f'Detected ImageFolder root: {VAL_ROOT}  (synsets: {len(syn)}, expect 1000)')
else:
    print('ERROR: synset folders not found; inspect /content/imagenet_val')
print('Final VAL_ROOT =', VAL_ROOT)


Dataset URL: https://www.kaggle.com/datasets/titericz/imagenet1k-val
License(s): CC0-1.0
Downloaded in 40s
Unzipping (50K images) ...
Unzipped in 25s
Detected ImageFolder root: /content/imagenet_val/imagenet-val  (synsets: 982, expect 1000)
Final VAL_ROOT = /content/imagenet_val/imagenet-val


## 3. 写入 IMC 代码 (%%writefile, 无需上传)

In [4]:
%%writefile imc_torch.py
#!/usr/bin/env python3
"""
imc_torch.py -- torch (GPU-capable) port of imc_vec.py's INT8 hybrid SRAM/PCM-IMC
forward for LeNet. Same math as the numpy reference (imc_vec.py); runs on cuda for
speed on Colab. Validated to reproduce imc_vec predictions image-for-image, and to
match the RTL golden bit-exactly at the no-drift/no-ADC/no-noise point.

dtype: float64 reproduces numpy exactly; float32 is fast on GPU and is exact for
LeNet layer sizes (all integer accumulations stay below 2^24). For deeper nets
(ImageNet) use float64 or int accumulation.
"""

import math
import torch
import numpy as np
try:
    from imc_vec import load_model, read_hex_bytes, LAYERS, WCOUNT  # LeNet-only; not used by ResNet
except Exception:
    load_model = read_hex_bytes = LAYERS = WCOUNT = None  # ResNet path uses imc_matmul_t directly

DEFAULT_CFG = {
    "cellBit": 2, "n_mag_cells": 4, "subArray": 64, "in_bits": 8,
    "adc_bits": None, "drift_t": 1.0, "comp": 0.0,
}


# ---------- move model to torch + precompute cells ----------
def to_torch_model(model, device="cpu", dtype=torch.float32, cellBit=2, ncell=4):
    cmax = (1 << cellBit) - 1
    tm = {"rshift": model["rshift"], "W": {}, "mask": {}, "nu": {}, "bias": {}, "cells": {}}
    for L in LAYERS:
        W = model["W"][L]                                  # numpy int (out,K)
        absW = np.abs(W)
        cells = np.stack([((absW >> (cellBit * j)) & cmax) for j in range(ncell)], axis=-1)
        tm["W"][L] = torch.tensor(W, device=device, dtype=dtype)
        tm["mask"][L] = torch.tensor(model["mask"][L], device=device)        # bool
        tm["nu"][L] = torch.tensor(model["nu"][L], device=device, dtype=dtype)
        tm["bias"][L] = torch.tensor(model["bias"][L], device=device, dtype=dtype)
        tm["cells"][L] = torch.tensor(cells, device=device, dtype=dtype)     # (out,K,ncell)
    return tm


def writenoise_torch(model, seed, device="cpu", dtype=torch.float32, cellBit=2, ncell=4):
    sigma_value = 0.03 * 127.0
    S = sum((2 ** (cellBit * j)) ** 2 for j in range(ncell))
    sc = sigma_value / math.sqrt(S)
    rng = np.random.RandomState(seed)
    return {L: torch.tensor(rng.normal(0.0, sc, model["W"][L].shape + (ncell,)),
                            device=device, dtype=dtype) for L in LAYERS}


def nu_torch(model, seed, device="cpu", dtype=torch.float32):
    rng = np.random.RandomState(seed)
    return {L: torch.tensor(rng.normal(0.10, 0.03, model["W"][L].shape),
                            device=device, dtype=dtype) for L in LAYERS}


# ---------- helpers ----------
def adc_quant_t(x, fs, bits):
    levels = (1 << bits) - 1
    safe = torch.where(fs > 0, fs, torch.ones_like(fs))
    q = torch.where(fs > 0, torch.round(x / safe * levels), torch.zeros_like(x))
    q = torch.clamp(q, 0, levels)
    return q / levels * fs


def requant_relu_t(acc, rshift):
    shifted = torch.floor(acc / (1 << rshift))
    return torch.where(acc <= 0, torch.zeros_like(acc), torch.clamp(shifted, max=127))


def requant_fc3_t(acc, rshift):
    shifted = torch.floor(acc / (1 << rshift))
    POS, NEG = 32767, -131071
    tr = shifted.to(torch.int64) & 0xFFFF
    tr = torch.where((tr & 0x8000) != 0, tr - 0x10000, tr).to(acc.dtype)
    res = torch.where(shifted > POS, torch.full_like(shifted, 32767.0),
                      torch.where(shifted < NEG, torch.full_like(shifted, -32768.0), tr))
    return res


def im2col_t(x, kh, kw, pad):
    B, C, H, W = x.shape
    OH, OW = H + 2 * pad - kh + 1, W + 2 * pad - kw + 1
    if pad:
        xp = torch.zeros((B, C, H + 2 * pad, W + 2 * pad), device=x.device, dtype=x.dtype)
        xp[:, :, pad:pad + H, pad:pad + W] = x
    else:
        xp = x
    cols = torch.empty((B, C, kh, kw, OH, OW), device=x.device, dtype=x.dtype)
    for ky in range(kh):
        for kx in range(kw):
            cols[:, :, ky, kx, :, :] = xp[:, :, ky:ky + OH, kx:kx + OW]
    return cols.reshape(B, C * kh * kw, OH * OW), OH, OW


# ---------- IMC matmul (torch) ----------
def imc_matmul_t(tm, L, V, cfg, kind, wn_L=None):
    cells_L, W_L, mask_L, nu_L, bias_L = tm["cells"][L], tm["W"][L], tm["mask"][L], tm["nu"][L], tm["bias"][L]
    rshift = tm["rshift"][L]
    device, dtype = W_L.device, W_L.dtype
    out_dim, K = W_L.shape
    cellBit, ncell, sub, in_bits = cfg["cellBit"], cfg["n_mag_cells"], cfg["subArray"], cfg["in_bits"]
    adc_bits, t, comp = cfg["adc_bits"], cfg["drift_t"], cfg["comp"]
    cmax = (1 << cellBit) - 1
    M = V.shape[1]
    zeroW = torch.zeros_like(W_L)

    Wp = torch.where(mask_L, W_L, zeroW)
    acc = bias_L[:, None] + Wp @ V

    Wu = torch.where(mask_L, zeroW, W_L)
    if not bool((Wu != 0).any()):
        # fully protected layer (e.g. conv1): no PCM contribution, skip the analog path
        if kind == "raw":
            return acc
        if kind == "fc3":
            return requant_fc3_t(acc, rshift)
        return requant_relu_t(acc, rshift)
    signpos = (Wu > 0).to(dtype)
    signneg = (Wu < 0).to(dtype)
    if t != 1.0:
        Dpcm = torch.pow(torch.as_tensor(float(t), device=device, dtype=dtype), -nu_L)
        D = torch.where(mask_L, torch.ones_like(nu_L), Dpcm)
    else:
        D = torch.ones_like(W_L)

    T = (K + sub - 1) // sub
    Kpad = T * sub
    actual_rows = torch.full((T,), float(sub), device=device, dtype=dtype)
    if Kpad != K:
        actual_rows[-1] = float(K - (T - 1) * sub)

    def padK(a):
        if Kpad == K:
            return a
        o = torch.zeros(a.shape[:-1] + (Kpad,), device=device, dtype=dtype)
        o[..., :K] = a
        return o

    Vpad = torch.zeros((Kpad, M), device=device, dtype=dtype)
    Vpad[:K, :] = V
    Vint = Vpad.to(torch.int64)
    Vbits = torch.stack([((Vint >> i) & 1).to(dtype) for i in range(in_bits)], dim=0)
    Vbits_t = Vbits.reshape(in_bits, T, sub, M)

    notmask = (~mask_L).to(dtype)
    pcm = torch.zeros((out_dim, M), device=device, dtype=dtype)
    for j in range(ncell):
        Cj = cells_L[:, :, j]
        if wn_L is not None:
            Cj = Cj + wn_L[:, :, j] * notmask
        CjD_pos = padK(Cj * D * signpos).reshape(out_dim, T, sub)
        CjD_neg = padK(Cj * D * signneg).reshape(out_dim, T, sub)
        wj = float(1 << (cellBit * j))
        for i in range(in_bits):
            Bi = Vbits_t[i]
            ppos = torch.einsum('trm,otr->otm', Bi, CjD_pos)
            pneg = torch.einsum('trm,otr->otm', Bi, CjD_neg)
            if adc_bits is not None:
                fs = (actual_rows * cmax)
                ppos = adc_quant_t(ppos, fs[None, :, None], adc_bits)
                pneg = adc_quant_t(pneg, fs[None, :, None], adc_bits)
            contrib = (ppos - pneg).sum(dim=1)
            bw = -(1 << i) if i == in_bits - 1 else (1 << i)
            pcm = pcm + (float(bw) * wj) * contrib
    if comp != 0.0:
        pcm = pcm * (math.exp(comp * math.log(t)) if t > 0 else 1.0)
    acc = acc + pcm

    if kind == "raw":
        return acc
    if kind == "fc3":
        return requant_fc3_t(acc, rshift)
    return requant_relu_t(acc, rshift)


# ---------- full LeNet forward (torch, batched) ----------
def forward_batch_t(images_int, tm, cfg, wn=None):
    device, dtype = tm["W"]["conv1"].device, tm["W"]["conv1"].dtype
    B = images_int.shape[0]
    gw = lambda L: (wn[L] if wn is not None else None)
    x = torch.tensor(images_int.reshape(B, 1, 28, 28), device=device, dtype=dtype)

    col, OH, OW = im2col_t(x, 5, 5, 2)
    V = col.permute(1, 0, 2).reshape(25, B * OH * OW)
    c1 = imc_matmul_t(tm, "conv1", V, cfg, "relu", gw("conv1"))
    c1 = c1.reshape(6, B, OH * OW).permute(1, 0, 2).reshape(B, 6, 28, 28)
    p1 = c1.reshape(B, 6, 14, 2, 14, 2).amax(dim=(3, 5))

    col, OH, OW = im2col_t(p1, 5, 5, 0)
    V = col.permute(1, 0, 2).reshape(150, B * OH * OW)
    c2 = imc_matmul_t(tm, "conv2", V, cfg, "relu", gw("conv2"))
    c2 = c2.reshape(16, B, OH * OW).permute(1, 0, 2).reshape(B, 16, 10, 10)
    p2 = c2.reshape(B, 16, 5, 2, 5, 2).amax(dim=(3, 5))
    flat = p2.reshape(B, 400)

    f1 = imc_matmul_t(tm, "fc1", flat.t().contiguous(), cfg, "relu", gw("fc1")).t()
    f2 = imc_matmul_t(tm, "fc2", f1.t().contiguous(), cfg, "relu", gw("fc2")).t()
    f3 = imc_matmul_t(tm, "fc3", f2.t().contiguous(), cfg, "fc3", gw("fc3")).t()
    preds = torch.argmax(f3, dim=1)
    return f3, preds


Writing imc_torch.py


In [5]:
%%writefile imc_resnet.py
#!/usr/bin/env python3
"""
imc_resnet.py -- ResNet-50 hybrid SRAM/PCM-IMC forward (analog crossbar + ADC).

Each PCM-mapped conv/linear (per config_imagenet.is_pcm_weight: weight, not BN,
not downsample.1) is replaced by an IMC module that:
  - quantizes its input activations to INT8 (per-layer calibrated scale),
  - runs the same validated IMC MAC as LeNet (imc_torch.imc_matmul_t, "raw"):
    cellBit=2 sign-magnitude cells, two's-complement bit-serial input, 64-row
    subarray tiling, 5-bit ADC per tile, per-weight drift t^(-nu), per-cell write
    noise; protected (top-k) weights computed exactly (SRAM),
  - dequantizes the integer MAC by (act_scale * weight_scale) to float; BN/ReLU/
    residual/pool stay in float (weight-only-quant baseline keeps activations fp,
    so between IMC convs we return to float exactly as that pipeline does).

mode='ref' replaces the IMC MAC with an exact INT8 conv (F.conv2d on the integer
tensors) so the IMC path can be validated bit-exactly at no-drift/no-ADC/no-noise.

Weights are quantized exactly as config_imagenet.build_quant_cache:
scale = w.abs().max()/127, w_q = round(w/scale). Protection top-k is by |w_orig|,
k = round(ratio*numel), matching config_imagenet.apply_drift_inplace.
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from imc_torch import imc_matmul_t

CELLBIT, NCELL = 2, 4
DRIFT_MEAN, DRIFT_STD, WRITE_NOISE = 0.10, 0.03, 0.03
BASE_CFG = {"cellBit": CELLBIT, "n_mag_cells": NCELL, "subArray": 64, "in_bits": 8,
            "adc_bits": None, "drift_t": 1.0, "comp": 0.0}


def writenoise_sigma_cell(cellBit=CELLBIT, ncell=NCELL):
    return WRITE_NOISE * 127.0 / math.sqrt(sum((2 ** (cellBit * j)) ** 2 for j in range(ncell)))


def is_pcm_weight(name):
    if "weight" not in name:
        return False
    if "bn" in name or "downsample.1" in name:
        return False
    return True


def _cells_from_wq(Wq2d, device, dtype):
    absW = Wq2d.abs().to(torch.int64)
    cmax = (1 << CELLBIT) - 1
    return torch.stack([((absW >> (CELLBIT * j)) & cmax).to(dtype)
                        for j in range(NCELL)], dim=-1)


class _IMCBase(nn.Module):
    """Holds INT8 weight + cells + per-run state shared by conv and linear."""
    def __init__(self, weight, device, dtype):
        super().__init__()
        self.device_, self.dtype_ = device, dtype
        w = weight.detach().to(torch.float32)
        self.w_orig = w.clone()
        wmax = w.abs().max()
        self.w_scale = float(wmax / 127.0) if wmax > 0 else 1.0
        wq4d = torch.round(w / (self.w_scale + 1e-9))                 # (out, ...)
        self.out_ch = wq4d.shape[0]
        self.Wq4d = wq4d.to(device=device, dtype=dtype)              # for ref conv
        self.Wq2d = wq4d.reshape(self.out_ch, -1).to(device=device, dtype=dtype)
        self.cells = _cells_from_wq(self.Wq2d, device, dtype)        # (out,K,ncell)
        self.numel = int(self.w_orig.numel())
        # per-run state
        self.act_scale = 1.0
        self.mask = None
        self.nu = None
        self.wn = None
        self.cfg = dict(BASE_CFG)
        self.mode = "imc"
        self.bias0 = torch.zeros(self.out_ch, device=device, dtype=dtype)

    def _mini_tm(self):
        return {"cells": {"x": self.cells}, "W": {"x": self.Wq2d}, "mask": {"x": self.mask},
                "nu": {"x": self.nu}, "bias": {"x": self.bias0}, "rshift": {"x": 0}}

    def set_state(self, act_scale, mask, nu, wn, cfg, mode="imc"):
        self.act_scale = float(act_scale)
        self.mask = mask
        self.nu = nu
        self.wn = wn
        self.cfg = cfg
        self.mode = mode

    def _effective_weight_noadc(self):
        """Effective INT-domain weight for the NO-ADC case, computed in one shot.
        Without an ADC the bit-serial/cell reconstruction is loss-free, so the
        crossbar output equals an exact MAC against this effective weight:
          unprotected: sign(Wq) * [sum_j (cell_j + wn_j) 4^j] * t^(comp - nu)
          protected:   Wq  (drift-free, no write noise)
        This is algebraically identical to imc_matmul_t with adc_bits=None
        (verified numerically), but costs one matmul instead of 32 einsums."""
        cw = torch.tensor([float(1 << (CELLBIT * j)) for j in range(NCELL)],
                          device=self.device_, dtype=self.dtype_)        # 4^j
        cellsn = self.cells + self.wn if self.wn is not None else self.cells
        mag = (cellsn * cw).sum(dim=-1)                                   # (out,K)
        sign = torch.sign(self.Wq2d)                                     # 0 for exact-zero weights
        t = self.cfg["drift_t"]; comp = self.cfg["comp"]
        if t != 1.0:
            Dpcm = torch.pow(torch.as_tensor(float(t), device=self.device_, dtype=self.dtype_),
                             comp - self.nu)                              # t^(comp - nu)
            D = torch.where(self.mask, torch.ones_like(self.nu), Dpcm)
        else:
            D = torch.ones_like(self.Wq2d)
        Weff_unprot = sign * mag * D
        return torch.where(self.mask, self.Wq2d, Weff_unprot)            # (out,K)


class IMCConv2d(_IMCBase):
    def __init__(self, conv, device, dtype):
        super().__init__(conv.weight, device, dtype)
        self.stride = conv.stride
        self.padding = conv.padding
        self.dilation = conv.dilation
        self.kh, self.kw = conv.kernel_size

    def forward(self, x):
        B, Cin, H, W = x.shape
        sH, sW = self.stride; pH, pW = self.padding; dH, dW = self.dilation
        OH = (H + 2 * pH - dH * (self.kh - 1) - 1) // sH + 1
        OW = (W + 2 * pW - dW * (self.kw - 1) - 1) // sW + 1
        xq = torch.round(x / (self.act_scale + 1e-9)).clamp(-128, 127).to(self.dtype_)
        deq = self.act_scale * self.w_scale
        if self.mode == "ref":
            out = F.conv2d(xq, self.Wq4d, None, self.stride, self.padding, self.dilation)
            return out * deq
        if self.mode == "imc_noadc":
            Weff = self._effective_weight_noadc().reshape(self.Wq4d.shape)
            out = F.conv2d(xq, Weff, None, self.stride, self.padding, self.dilation)
            return out * deq
        V = F.unfold(xq, (self.kh, self.kw), self.dilation, self.padding, self.stride)  # (B,K,L)
        K = V.shape[1]
        Vk = V.permute(1, 0, 2).reshape(K, B * OH * OW)
        acc = imc_matmul_t(self._mini_tm(), "x", Vk, self.cfg, "raw", self.wn)          # (out, B*L)
        out = acc.reshape(self.out_ch, B, OH * OW).permute(1, 0, 2).reshape(B, self.out_ch, OH, OW)
        return out * deq


class IMCLinear(_IMCBase):
    def __init__(self, lin, device, dtype):
        super().__init__(lin.weight, device, dtype)
        self.lin_bias = None if lin.bias is None else lin.bias.detach().to(device=device, dtype=dtype)

    def forward(self, x):
        xq = torch.round(x / (self.act_scale + 1e-9)).clamp(-128, 127).to(self.dtype_)
        deq = self.act_scale * self.w_scale
        if self.mode == "ref":
            out = xq @ self.Wq2d.t()
        elif self.mode == "imc_noadc":
            out = xq @ self._effective_weight_noadc().t()
        else:
            acc = imc_matmul_t(self._mini_tm(), "x", xq.t().contiguous(), self.cfg, "raw", self.wn)
            out = acc.t()
        out = out * deq
        if self.lin_bias is not None:
            out = out + self.lin_bias
        return out


# ---------- module tree surgery ----------
def _get_parent(model, path):
    parts = path.split(".")
    obj = model
    for p in parts[:-1]:
        obj = obj[int(p)] if p.isdigit() else getattr(obj, p)
    return obj, parts[-1]


def _set_module(model, path, new):
    parent, last = _get_parent(model, path)
    if last.isdigit():
        parent[int(last)] = new
    else:
        setattr(parent, last, new)


def build_imc_resnet(model, device="cuda", dtype=torch.float32):
    """Swap every PCM-mapped conv/linear with an IMC module. Returns {path: imc_module}."""
    imc = {}
    pcm_names = [n for n, _ in model.named_parameters() if is_pcm_weight(n)]
    mods = dict(model.named_modules())
    for n in pcm_names:
        path = n[:-len(".weight")]
        m = mods[path]
        new = IMCLinear(m, device, dtype) if isinstance(m, nn.Linear) else IMCConv2d(m, device, dtype)
        _set_module(model, path, new)
        imc[path] = new
    model.to(device).eval()
    return imc


# ---------- activation-scale calibration ----------
def calibrate_act_scales(imc_modules, model, calib_batches, device):
    """Run a few fp batches in 'ref' mode and record per-layer input max-abs.
    (ref mode quantizes with the CURRENT scale but we only need the input range;
    we set scale=1 first so xq=round(x) does not clip the range estimate.)"""
    maxabs = {p: 0.0 for p in imc_modules}
    handles = []
    for p, m in imc_modules.items():
        def hook(mod, inp, _p=p):
            v = inp[0].detach().abs().max().item()
            if v > maxabs[_p]:
                maxabs[_p] = v
        handles.append(m.register_forward_pre_hook(hook))
    # temporary identity-ish state so the model runs (ref mode, scale 1, no clip issue
    # because ref path computes F.conv2d on round(x); fine for range collection)
    for m in imc_modules.values():
        m.set_state(1.0, torch.zeros(m.Wq2d.shape, dtype=torch.bool, device=device),
                    None, None, dict(BASE_CFG), mode="ref")
    with torch.no_grad():
        for xb in calib_batches:
            model(xb.to(device))
    for h in handles:
        h.remove()
    return {p: (maxabs[p] / 127.0 if maxabs[p] > 0 else 1.0) for p in imc_modules}


# ---------- protection masks (top-k by |w_orig|, matching config_imagenet) ----------
def make_masks(imc_modules, ratios, device):
    masks = {}
    for p, m in imc_modules.items():
        r = float(ratios.get(p + ".weight", ratios.get(p, 0.0)))
        k = max(0, min(int(round(r * m.numel)), m.numel))
        flat = m.w_orig.abs().flatten()
        mk = torch.zeros(m.numel, dtype=torch.bool)
        if k > 0:
            idx = torch.topk(flat, k, largest=True, sorted=False).indices
            mk[idx] = True
        masks[p] = mk.view(m.Wq2d.shape).to(device)
    return masks


def uniform_ratios(imc_modules, budget=0.05):
    return {p: budget for p in imc_modules}


def none_ratios(imc_modules):
    return {p: 0.0 for p in imc_modules}


# ---------- per-seed drift / write-noise ----------
def gen_nu(imc_modules, seed, device, dtype):
    rng = np.random.RandomState(seed)
    return {p: torch.tensor(rng.normal(DRIFT_MEAN, DRIFT_STD, m.Wq2d.shape),
                            device=device, dtype=dtype) for p, m in imc_modules.items()}


def gen_wn(imc_modules, seed, device, dtype):
    sc = writenoise_sigma_cell()
    rng = np.random.RandomState(seed)
    return {p: torch.tensor(rng.normal(0.0, sc, m.Wq2d.shape + (NCELL,)),
                            device=device, dtype=dtype) for p, m in imc_modules.items()}


def set_run_state(imc_modules, act_scales, masks, nu, wn, drift_t, adc_bits, comp, mode="imc"):
    cfg = dict(BASE_CFG)
    cfg["drift_t"] = drift_t; cfg["adc_bits"] = adc_bits; cfg["comp"] = comp
    for p, m in imc_modules.items():
        m.set_state(act_scales[p], masks[p], nu[p] if nu else None,
                    wn[p] if wn else None, cfg, mode=mode)


Writing imc_resnet.py


In [6]:
%%writefile run_resnet_imc.py
#!/usr/bin/env python3
"""
run_resnet_imc.py -- ResNet-50 hybrid SRAM/PCM-IMC accuracy sweep on an ImageNet
val subset (H100). Same IMC model as the LeNet experiment (cellBit=2 sign-magnitude,
two's-complement bit-serial input, 64-row tiling, 5-bit ADC, per-weight drift,
per-cell write noise; protected top-k computed exactly).

Arms: none / uniform(beta=5%) / layer_aware(Table IX ratios).
For each arm: a 'clean' point (full INT8, no drift/ADC/noise, via the exact-INT8
ref path) plus a sweep over {no-ADC, 5-bit ADC} x drift-grid x seeds.

The storage-drift Paper-1 baseline (76.14%) is weight-only INT8 with fp activations;
this experiment is full INT8 (weights AND activations, since the IMC crossbar needs
a digital input), so the clean point sits a little below 76.14% by the activation
-quant penalty. The comparison that matters is within this experiment, across arms.

Subset is a fixed seeded random sample of the 50000 val images (deterministic).
Run --timing_only first to size n_images / drift_grid / seeds for the full run.
"""

import argparse, json, time, platform
import numpy as np
import torch

import imc_resnet as R


# ---------------- core (model/loader-agnostic, unit-testable) ----------------
def float_calibrate(model, pcm_paths, loader, device, n_batches):
    """Per-layer input-activation scale = max-abs / 127, measured on the FLOAT
    model (before IMC swap) over a few batches. This is the activation
    quantization scale the IMC crossbar input uses."""
    import torch.nn as nn
    mods = dict(model.named_modules())
    maxabs = {p: 0.0 for p in pcm_paths}
    handles = []
    for p in pcm_paths:
        def hook(mod, inp, _p=p):
            v = inp[0].detach().abs().max().item()
            if v > maxabs[_p]:
                maxabs[_p] = v
        handles.append(mods[p].register_forward_pre_hook(hook))
    model.eval()
    with torch.no_grad():
        for b, (x, _) in enumerate(loader):
            model(x.to(device))
            if b + 1 >= n_batches:
                break
    for h in handles:
        h.remove()
    return {p: (maxabs[p] / 127.0 if maxabs[p] > 0 else 1.0) for p in pcm_paths}


@torch.no_grad()
def eval_loader(model, loader, device):
    c1 = c5 = tot = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        out = model(x)
        _, p5 = out.topk(5, 1, True, True)
        corr = p5.eq(y.view(-1, 1))
        c1 += corr[:, 0].sum().item()
        c5 += corr.any(1).sum().item()
        tot += y.size(0)
    return c1 / tot, c5 / tot


def build_arms(imc, ratios, device, budget=0.05):
    return {
        "none":        R.make_masks(imc, R.none_ratios(imc), device),
        "uniform":     R.make_masks(imc, R.uniform_ratios(imc, budget), device),
        "layer_aware": R.make_masks(imc, ratios, device),
    }


def run_config(model, imc, loader, device, dt, act_scales, mask,
               drift_t, adc_bits, seed, wn_on):
    """One full-subset evaluation under a given (mask, drift_t, adc, seed)."""
    nu = R.gen_nu(imc, 1000 + seed, device, dt) if drift_t != 1.0 else None
    wn = R.gen_wn(imc, 5000 + seed, device, dt) if wn_on else None
    comp = R.DRIFT_MEAN if drift_t != 1.0 else 0.0
    mode = "imc_noadc" if adc_bits is None else "imc"   # no-ADC is loss-free => one matmul
    R.set_run_state(imc, act_scales, mask, nu, wn, drift_t, adc_bits, comp, mode=mode)
    return eval_loader(model, loader, device)


def sweep(model, imc, loader, device, dt, act_scales, arms,
          drift_grid, seeds, wn_on, verbose=True, out_path=None, meta=None):
    runs = []

    def checkpoint():
        if out_path is not None:
            json.dump({"meta": meta, "runs": runs, "complete": False}, open(out_path, "w"), indent=2)

    # clean per arm: exact INT8 (ref path), no drift / ADC / noise
    for arm, mask in arms.items():
        R.set_run_state(imc, act_scales, mask, None, None, 1.0, None, 0.0, mode="ref")
        a1, a5 = eval_loader(model, loader, device)
        runs.append({"arm": arm, "kind": "clean", "drift_t": 1.0, "adc_bits": None,
                     "top1": a1, "top5": a5})
        checkpoint()
        if verbose:
            print(f"  clean {arm:12s} top1={a1:.4f} top5={a5:.4f}", flush=True)
    # sweep: arm x adc x drift x seed
    for arm, mask in arms.items():
        for adc in (None, 5):
            for t in drift_grid:
                t1s, t5s = [], []
                for s in range(seeds):
                    a1, a5 = run_config(model, imc, loader, device, dt, act_scales,
                                        mask, t, adc, s, wn_on)
                    t1s.append(a1); t5s.append(a5)
                rec = {"arm": arm, "kind": "sweep", "drift_t": t,
                       "adc_bits": adc, "seeds": list(range(seeds)),
                       "top1_seeds": t1s, "top5_seeds": t5s,
                       "top1_mean": float(np.mean(t1s)), "top1_std": float(np.std(t1s)),
                       "top5_mean": float(np.mean(t5s)), "top5_std": float(np.std(t5s))}
                runs.append(rec)
                checkpoint()
                if verbose:
                    print(f"  {arm:12s} adc={str(adc):4s} t={t:>8.0f} "
                          f"top1={rec['top1_mean']:.4f}±{rec['top1_std']:.4f}", flush=True)
    return runs


# ---------------- ImageNet wiring (used on Colab; not exercised in sandbox) ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def imagenet_subset_loader(val_root, n_images, subset_seed, batch_size, num_workers):
    from torchvision import datasets, transforms
    tx = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                             transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    ds = datasets.ImageFolder(val_root, transform=tx)
    rng = np.random.RandomState(subset_seed)
    idx = np.sort(rng.choice(len(ds), size=min(n_images, len(ds)), replace=False))
    sub = torch.utils.data.Subset(ds, idx.tolist())
    return torch.utils.data.DataLoader(sub, batch_size=batch_size, shuffle=False,
                                       num_workers=num_workers, pin_memory=True), len(sub)


def main():
    from torchvision import models
    ap = argparse.ArgumentParser()
    ap.add_argument("--val_root", required=True)
    ap.add_argument("--ratios_json", required=True)
    ap.add_argument("--out", default="resnet_imc_results.json")
    ap.add_argument("--n_images", type=int, default=2000)
    ap.add_argument("--subset_seed", type=int, default=42)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--num_workers", type=int, default=4)
    ap.add_argument("--calib_batches", type=int, default=4)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--drift_grid", default="1,1e2,1e4,1e6")
    ap.add_argument("--no_write_noise", action="store_true")
    ap.add_argument("--dtype", default="float32")
    ap.add_argument("--timing_only", action="store_true")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dt = torch.float64 if args.dtype == "float64" else torch.float32
    drift_grid = [float(s) for s in args.drift_grid.split(",")]
    ratios = json.load(open(args.ratios_json))
    wn_on = not args.no_write_noise

    loader, nsub = imagenet_subset_loader(args.val_root, args.n_images, args.subset_seed,
                                          args.batch_size, args.num_workers)
    print(f"subset images: {nsub}  device: {device}  dtype: {dt}  batch: {args.batch_size}", flush=True)

    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).eval().to(dt).to(device)
    pcm_paths = [n[:-len(".weight")] for n, _ in model.named_parameters() if R.is_pcm_weight(n)]
    print("calibrating activation scales (float model) ...", flush=True)
    act_scales = float_calibrate(model, pcm_paths, loader, device, args.calib_batches)

    imc = R.build_imc_resnet(model, device=device, dtype=dt)
    arms = build_arms(imc, ratios, device)

    if args.timing_only:
        torch.cuda.synchronize() if device == "cuda" else None
        t0 = time.time()
        a1, a5 = run_config(model, imc, loader, device, dt, act_scales,
                            arms["layer_aware"], 1e6, 5, 0, wn_on)
        torch.cuda.synchronize() if device == "cuda" else None
        el = time.time() - t0
        print(f"[timing] one full-subset config (layer_aware, t=1e6, 5-bit ADC): "
              f"{el:.1f}s  =>  {el / nsub * 1000:.1f} ms/img  top1={a1:.4f}", flush=True)
        n_cfg = 3 + 3 * 2 * len(drift_grid) * args.seeds
        print(f"[timing] full sweep ~= {n_cfg} configs => est {el * n_cfg / 60:.0f} min "
              f"(clean is cheaper; this is an upper bound)", flush=True)
        return

    meta = {"model": "resnet50", "n_images": nsub, "subset_seed": args.subset_seed,
            "batch_size": args.batch_size, "seeds": args.seeds, "drift_grid": drift_grid,
            "write_noise": wn_on, "dtype": str(dt), "cellBit": R.CELLBIT,
            "n_mag_cells": R.NCELL, "adc": "5-bit (NeuroSim levelOutput=32)",
            "scheme": "2-bit sign-magnitude (III)",
            "sigma_cell": R.writenoise_sigma_cell(),
            "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
            "torch": torch.__version__}
    print("starting sweep ...", flush=True)
    t0 = time.time()
    runs = sweep(model, imc, loader, device, dt, act_scales, arms,
                 drift_grid, args.seeds, wn_on, out_path=args.out, meta=meta)
    meta["elapsed_sec"] = time.time() - t0
    json.dump({"meta": meta, "runs": runs, "complete": True}, open(args.out, "w"), indent=2)
    print(f"\nwrote {args.out}  ({meta['elapsed_sec']/60:.1f} min)", flush=True)


if __name__ == "__main__":
    main()


Writing run_resnet_imc.py


## 4. 路径检查 + GPU 确认

In [7]:
import json, torch
RATIOS_JSON = f'{PROJECT}/outputs_ResNet50_ImageNet/LayerAware_Ratios_ResNet50.json'
r = json.load(open(RATIOS_JSON))
print('ratios entries:', len(r), '(expect 54)')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
OUT_JSON = f'{PROJECT}/outputs_ResNet50_ImageNet/resnet_imc_results.json'
print('results will be written to:', OUT_JSON)


ratios entries: 54 (expect 54)
torch 2.11.0+cu128 | cuda True | NVIDIA RTX PRO 6000 Blackwell Server Edition
results will be written to: /content/drive/MyDrive/PCM_Research7/outputs_ResNet50_ImageNet/resnet_imc_results.json


## 5. 计时探针 (先跑这个)
跑一个最慢的配置(layer_aware, t=1e6, 5-bit ADC)在 512 张上,报 ms/img 和整轮扫描的估算时长。看完再决定下面正式跑的 `n_images` 和网格。

In [8]:
!python run_resnet_imc.py --timing_only \
    --val_root {VAL_ROOT} --ratios_json {RATIOS_JSON} \
    --n_images 512 --batch_size 64


subset images: 512  device: cuda  dtype: torch.float32  batch: 64
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:03<00:00, 28.0MB/s]
calibrating activation scales (float model) ...
[timing] one full-subset config (layer_aware, t=1e6, 5-bit ADC): 104.9s  =>  204.9 ms/img  top1=0.0000
[timing] full sweep ~= 75 configs => est 131 min (clean is cheaper; this is an upper bound)


In [10]:
from google.colab import files
files.upload()

Saving LayerAware_Ratios_ResNet50_beta50.json to LayerAware_Ratios_ResNet50_beta50.json


{'LayerAware_Ratios_ResNet50_beta50.json': b'{\n  "conv1.weight": 1.0,\n  "layer1.0.conv1.weight": 1.0,\n  "layer1.0.conv2.weight": 1.0,\n  "layer1.0.conv3.weight": 1.0,\n  "layer1.0.downsample.0.weight": 1.0,\n  "layer1.1.conv1.weight": 1.0,\n  "layer1.1.conv2.weight": 1.0,\n  "layer1.1.conv3.weight": 1.0,\n  "layer1.2.conv1.weight": 1.0,\n  "layer1.2.conv2.weight": 1.0,\n  "layer1.2.conv3.weight": 1.0,\n  "layer2.0.conv1.weight": 1.0,\n  "layer2.0.conv2.weight": 1.0,\n  "layer2.0.conv3.weight": 1.0,\n  "layer2.0.downsample.0.weight": 1.0,\n  "layer2.1.conv1.weight": 0.3139224542532097,\n  "layer2.1.conv2.weight": 1.0,\n  "layer2.1.conv3.weight": 1.0,\n  "layer2.2.conv1.weight": 0.7481958762228557,\n  "layer2.2.conv2.weight": 0.7851334304171103,\n  "layer2.2.conv3.weight": 1.0,\n  "layer2.3.conv1.weight": 0.8811710713220798,\n  "layer2.3.conv2.weight": 0.947845694462583,\n  "layer2.3.conv3.weight": 1.0,\n  "layer3.0.conv1.weight": 1.0,\n  "layer3.0.conv2.weight": 0.3040100979378327,\n

In [11]:
# 5b. 快速诊断:真实权重下确认管线精度正常 + 看清退化曲线(只跑 layer_aware, 512 张, 约 4-5 分钟)
import json, time, torch
import imc_resnet as R, run_resnet_imc as RUN
from torchvision import models

DIAG_N = 512
DT = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

ratios = json.load(open('/content/LayerAware_Ratios_ResNet50_beta50.json'))
loader, nsub = RUN.imagenet_subset_loader(VAL_ROOT, DIAG_N, 42, 64, 4)
print(f'diag images: {nsub}  device: {device}\n')

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).eval().to(DT).to(device)
pcm = [n[:-len('.weight')] for n, _ in model.named_parameters() if R.is_pcm_weight(n)]
acts = RUN.float_calibrate(model, pcm, loader, device, 4)
imc = R.build_imc_resnet(model, device=device, dtype=DT)
la = R.make_masks(imc, ratios, device)

def run(tag, drift_t, adc, mode):
    nu = R.gen_nu(imc, 1000, device, DT) if drift_t != 1.0 else None
    wn = R.gen_wn(imc, 5000, device, DT)
    comp = R.DRIFT_MEAN if drift_t != 1.0 else 0.0
    R.set_run_state(imc, acts, la, nu, wn, drift_t, adc, comp, mode=mode)
    t0 = time.time(); a1, a5 = RUN.eval_loader(model, loader, device)
    print(f'  {tag:30s} top1={a1*100:5.1f}%  top5={a5*100:5.1f}%   ({time.time()-t0:.0f}s)')

# (1) clean: 全 INT8, 无漂移/ADC/噪声 —— 关键判据, 应 70-75%
R.set_run_state(imc, acts, la, None, None, 1.0, None, 0.0, mode='ref')
t0 = time.time(); c1, c5 = RUN.eval_loader(model, loader, device)
print(f'  {"CLEAN (INT8)":30s} top1={c1*100:5.1f}%  top5={c5*100:5.1f}%   ({time.time()-t0:.0f}s)')
print('  ^ 应在 70-75%。若接近 0% = 有 bug, 停下把本段输出发我。\n')

# (2) 漂移阶梯, 无 ADC(快)
print('  --- drift ladder, no ADC ---')
for t in (1.0, 1e2, 1e4, 1e6):
    run(f'no-ADC  t={int(t)}', t, None, 'imc_noadc')
print()

# (3) 5-bit ADC, 两端(慢)
print('  --- 5-bit ADC (two ends) ---')
run('5-bit   t=1        (ADC only)', 1.0, 5, 'imc')
run('5-bit   t=1000000  (worst)', 1e6, 5, 'imc')

diag images: 512  device: cuda

  CLEAN (INT8)                   top1= 72.7%  top5= 91.2%   (1s)
  ^ 应在 70-75%。若接近 0% = 有 bug, 停下把本段输出发我。

  --- drift ladder, no ADC ---
  no-ADC  t=1                    top1= 68.2%  top5= 89.1%   (1s)
  no-ADC  t=100                  top1= 67.6%  top5= 90.0%   (1s)
  no-ADC  t=10000                top1= 66.8%  top5= 88.7%   (1s)
  no-ADC  t=1000000              top1= 64.3%  top5= 87.5%   (1s)

  --- 5-bit ADC (two ends) ---
  5-bit   t=1        (ADC only)  top1= 67.8%  top5= 90.0%   (56s)
  5-bit   t=1000000  (worst)     top1= 47.1%  top5= 73.8%   (56s)


In [12]:
!cp /content/LayerAware_Ratios_ResNet50_beta50.json {PROJECT}/outputs_ResNet50_ImageNet/
!ls -la {PROJECT}/outputs_ResNet50_ImageNet/LayerAware_Ratios_ResNet50_beta50.json

-rw------- 1 root root 2134 Jun 11 15:17 /content/drive/MyDrive/PCM_Research7/outputs_ResNet50_ImageNet/LayerAware_Ratios_ResNet50_beta50.json


In [13]:
p = '/content/run_resnet_imc.py'
s = open(p).read()
s = s.replace('def build_arms(imc, ratios, device, budget=0.05):',
              'def build_arms(imc, ratios, device, budget=0.5):')
open(p, 'w').write(s)
assert 'def build_arms(imc, ratios, device, budget=0.5):' in open(p).read(), '补丁没匹配上,告诉我'
print('uniform 预算已设为 0.5,三条臂现在都是 beta=50%')

uniform 预算已设为 0.5,三条臂现在都是 beta=50%


## 6. 正式扫描
默认:5000 张子集、3 个 seed、漂移网格 1/1e2/1e4/1e6、batch 64。

如果第 5 步估算太长:把 `--n_images` 降到 2000,或把 `--drift_grid` 减成 `1,1e3,1e6`。如果 OOM:把 `--batch_size` 降到 32。

结果增量写到 Drive 的 `OUT_JSON`,中途断线也能保住已完成部分(重跑会覆盖重来)。

In [14]:
!python run_resnet_imc.py \
    --val_root {VAL_ROOT} \
    --ratios_json {PROJECT}/outputs_ResNet50_ImageNet/LayerAware_Ratios_ResNet50_beta50.json \
    --n_images 2000 --subset_seed 42 --seeds 3 \
    --drift_grid 1,1e2,1e4,1e6 --batch_size 64 \
    --out {OUT_JSON}

subset images: 2000  device: cuda  dtype: torch.float32  batch: 64
calibrating activation scales (float model) ...
starting sweep ...
  clean none         top1=0.7645 top5=0.9295
  clean uniform      top1=0.7645 top5=0.9295
  clean layer_aware  top1=0.7645 top5=0.9295
  none         adc=None t=       1 top1=0.0610±0.0130
  none         adc=None t=     100 top1=0.0303±0.0082
  none         adc=None t=   10000 top1=0.0027±0.0010
  none         adc=None t= 1000000 top1=0.0012±0.0006
  none         adc=5    t=       1 top1=0.0008±0.0002
  none         adc=5    t=     100 top1=0.0008±0.0005
  none         adc=5    t=   10000 top1=0.0008±0.0005
  none         adc=5    t= 1000000 top1=0.0010±0.0004
  uniform      adc=None t=       1 top1=0.5457±0.0086
  uniform      adc=None t=     100 top1=0.5272±0.0098
  uniform      adc=None t=   10000 top1=0.4718±0.0120
  uniform      adc=None t= 1000000 top1=0.3347±0.0097
  uniform      adc=5    t=       1 top1=0.7182±0.0021
  uniform      adc=5    t=   

## 7. 查看结果 + 下载

In [15]:
import json
d = json.load(open(OUT_JSON))
print('complete:', d.get('complete'), '| images:', d['meta']['n_images'],
      '| seeds:', d['meta']['seeds'], '| elapsed:', round(d['meta'].get('elapsed_sec',0)/60,1),'min')
print()
clean = {r['arm']: r['top1'] for r in d['runs'] if r['kind']=='clean'}
print('clean (full INT8) top1:', {k: round(v,4) for k,v in clean.items()})
print()
grid = sorted({r['drift_t'] for r in d['runs'] if r['kind']=='sweep'})
for adc in (None, 5):
    print(f'--- {"no-ADC" if adc is None else "5-bit ADC"} : top1 mean+/-std ---')
    hdr = 'arm'.ljust(13) + ''.join(f't={int(t):>10}' for t in grid)
    print(hdr)
    for arm in ('none','uniform','layer_aware'):
        row = arm.ljust(13)
        for t in grid:
            rec = next((r for r in d['runs'] if r['kind']=='sweep' and r['arm']==arm and r['adc_bits']==adc and r['drift_t']==t), None)
            row += (f"{rec['top1_mean']*100:>5.1f}+/-{rec['top1_std']*100:<4.1f}" if rec else ' '*10)
        print(row)
    print()
from google.colab import files
files.download(OUT_JSON)


complete: True | images: 2000 | seeds: 3 | elapsed: 224.6 min

clean (full INT8) top1: {'none': 0.7645, 'uniform': 0.7645, 'layer_aware': 0.7645}

--- no-ADC : top1 mean+/-std ---
arm          t=         1t=       100t=     10000t=   1000000
none           6.1+/-1.3   3.0+/-0.8   0.3+/-0.1   0.1+/-0.1 
uniform       54.6+/-0.9  52.7+/-1.0  47.2+/-1.2  33.5+/-1.0 
layer_aware   72.0+/-0.4  72.0+/-0.9  71.0+/-0.6  69.4+/-0.6 

--- 5-bit ADC : top1 mean+/-std ---
arm          t=         1t=       100t=     10000t=   1000000
none           0.1+/-0.0   0.1+/-0.0   0.1+/-0.0   0.1+/-0.0 
uniform       71.8+/-0.2  69.7+/-1.2  64.9+/-1.9  64.4+/-1.0 
layer_aware   72.8+/-0.1  71.0+/-0.4  64.9+/-0.3  52.5+/-0.8 



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/PCM_Research7'
!ls {PROJECT}


Mounted at /content/drive
config_imagenet.py			  Step_B1_Aggregate.py
kaggle.json				  Step_B1_LayerSensitivity.py
outputs_ResNet50_ImageNet		  Step_B2_BudgetSweep.py
PCM_Research5_Data_Step3_Sensitivity.csv  Step_B3_LongTerm.py
PCM_Research6_Data_Step3_Sensitivity.csv  Step_B4_Mismatch.py
README.md				  Step_B5_Scaling.py
Run_on_Colab.ipynb			  Step_B6_PrecisionAblation.py
Step_B0_VerifyBaseline.py		  test_dry_run.py


In [3]:
import os, time, glob
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/PCM_Research7/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

VAL_ROOT = '/content/imagenet_val'
if not (os.path.exists(VAL_ROOT) and len([d for d in os.listdir(VAL_ROOT) if d.startswith('n0')]) >= 100):
    !mkdir -p {VAL_ROOT}
    print('Downloading ImageNet val from Kaggle ...')
    !kaggle datasets download -d titericz/imagenet1k-val -p /content --quiet
    print('Unzipping 50K images ...')
    !unzip -q /content/imagenet1k-val.zip -d {VAL_ROOT}
    !rm -f /content/imagenet1k-val.zip

# 数据集可能有嵌套，自动定位真正的 ImageFolder 根目录
cand = glob.glob(f'{VAL_ROOT}/**/n01440764', recursive=True)
VAL_ROOT = os.path.dirname(cand[0]) if cand else VAL_ROOT
synsets = [d for d in os.listdir(VAL_ROOT) if d.startswith('n0')]
print(f'\nVAL_ROOT = {VAL_ROOT}   (synset 数: {len(synsets)}, 应为 1000)')

Dataset URL: https://www.kaggle.com/datasets/titericz/imagenet1k-val
License(s): CC0-1.0
Unzipping 50K images ...

VAL_ROOT = /content/imagenet_val/imagenet-val   (synset 数: 982, 应为 1000)


In [4]:
import os
alln = sorted(d for d in os.listdir(VAL_ROOT) if d.startswith('n'))
print('总类数 (n*):', len(alln), ' | n0:', sum(d.startswith('n0') for d in alln),
      ' n1:', sum(d.startswith('n1') for d in alln))

总类数 (n*): 1000  | n0: 982  n1: 18


In [2]:
from google.colab import files
files.upload()

Saving p1_wn_ablation.py to p1_wn_ablation.py


{'p1_wn_ablation.py': b'"""\nPaper 1 (storage pipeline) write-noise ablation.\n\nQuestion: is the layer-aware vs uniform advantage at high drift driven by\nDRIFT or by WRITE NOISE?  Runs Paper 1\'s OWN apply_drift_inplace (config_imagenet.py)\nwith write noise ON vs OFF at a fixed t, for algo_only / uniform / layer_aware.\n\nUses the existing `write_noise_override` argument (0.0 turns write noise off).\nNo change to config_imagenet.py.\n\nRun from the PCM_Research7 folder (next to config_imagenet.py), e.g.:\n    !python3 p1_wn_ablation.py --val_root {VAL_ROOT}\nFor a quick look (a few hundred images): add  --max_batches 4\n"""\nimport argparse, time\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom config_imagenet import (\n    load_resnet50, get_imagenet_val_loader, evaluate,\n    build_quant_cache, apply_drift_inplace, restore_baseline,\n    get_seed, stable_hash, out_path,\n    make_layer_aware_ratios, make_uniform_ratios, make_no_protection,\n    DRIFT_MEAN, TOTAL_BUDG

In [4]:
import shutil
shutil.copy('/content/p1_wn_ablation.py',
            '/content/drive/MyDrive/PCM_Research7/p1_wn_ablation.py')
%cd /content/drive/MyDrive/PCM_Research7
!python3 p1_wn_ablation.py --val_root {VAL_ROOT} --max_batches 8

/content/drive/MyDrive/PCM_Research7
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 345MB/s]
Traceback (most recent call last):
  File "/content/drive/MyDrive/PCM_Research7/p1_wn_ablation.py", line 119, in <module>
    main()
  File "/content/drive/MyDrive/PCM_Research7/p1_wn_ablation.py", line 62, in main
    val_loader = get_imagenet_val_loader(
                 ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/PCM_Research7/config_imagenet.py", line 121, in get_imagenet_val_loader
    val_ds = datasets.ImageFolder(val_root, transform=transform)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 328, in __init__
    super().__init__(
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 149, in __init__
    classes, class_to_idx = se

In [5]:
import os, glob

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/PCM_Research7/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

VAL_ROOT = '/content/imagenet_val'
if not (os.path.exists(VAL_ROOT) and len([d for d in os.listdir(VAL_ROOT) if d.startswith('n')]) >= 100):
    !mkdir -p {VAL_ROOT}
    print('Downloading ImageNet val ...')
    !kaggle datasets download -d titericz/imagenet1k-val -p /content --quiet
    print('Unzipping ...')
    !unzip -q /content/imagenet1k-val.zip -d {VAL_ROOT}
    !rm -f /content/imagenet1k-val.zip

cand = glob.glob(f'{VAL_ROOT}/**/n01440764', recursive=True)
VAL_ROOT = os.path.dirname(cand[0]) if cand else VAL_ROOT
print('VAL_ROOT =', VAL_ROOT)

Dataset URL: https://www.kaggle.com/datasets/titericz/imagenet1k-val
License(s): CC0-1.0
Unzipping ...
VAL_ROOT = /content/imagenet_val/imagenet-val


In [6]:
%cd /content/drive/MyDrive/PCM_Research7
!python3 p1_wn_ablation.py --val_root {VAL_ROOT} --max_batches 8

/content/drive/MyDrive/PCM_Research7
device cuda  t=1e+06  budget 0.5  protected fp32  amp True  eval 8 batches
/content/drive/MyDrive/PCM_Research7/config_imagenet.py:325: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
  algo_only    WNon   t=1e+06  top1=  0.00%  top5=  0.00%
  algo_only    WNoff  t=1e+06  top1=  0.00%  top5=  0.00%
  uniform      WNon   t=1e+06  top1= 43.07%  top5= 63.67%
  uniform      WNoff  t=1e+06  top1= 88.38%  top5= 97.46%
  layer_aware  WNon   t=1e+06  top1= 83.50%  top5= 94.34%
  layer_aware  WNoff  t=1e+06  top1= 89.06%  top5= 97.17%

=== Paper1 storage pipeline, t=1e+06, protected=fp32, amp=True ===
         arm |   WN on  WN off |  WN cost
   algo_only |    0.00    0.00 |    +0.00
     uniform |   43.07   88.38 |   +45.31
 layer_aware |   83.50   89.06 |    +5.57

LA - uniform  (WN on) : +40.43
LA - uniform  (WN off): +0.68   <--

In [7]:
!python3 p1_wn_ablation.py --val_root {VAL_ROOT}

device cuda  t=1e+06  budget 0.5  protected fp32  amp True  eval full-val
/content/drive/MyDrive/PCM_Research7/config_imagenet.py:325: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
  algo_only    WNon   t=1e+06  top1=  0.09%  top5=  0.54%
  algo_only    WNoff  t=1e+06  top1=  0.22%  top5=  1.07%
  uniform      WNon   t=1e+06  top1= 27.73%  top5= 51.06%
  uniform      WNoff  t=1e+06  top1= 74.73%  top5= 92.18%
  layer_aware  WNon   t=1e+06  top1= 68.98%  top5= 89.04%
  layer_aware  WNoff  t=1e+06  top1= 75.35%  top5= 92.65%

=== Paper1 storage pipeline, t=1e+06, protected=fp32, amp=True ===
         arm |   WN on  WN off |  WN cost
   algo_only |    0.09    0.22 |    +0.13
     uniform |   27.73   74.73 |   +47.00
 layer_aware |   68.98   75.35 |    +6.36

LA - uniform  (WN on) : +41.25
LA - uniform  (WN off): +0.62   <-- if ~0, Paper 1's advantage is write-n

In [6]:
!python3 comp_diag2_standalone.py --val_root {VAL_ROOT}

subset 500  device cuda  budget 0.5  drift [1000000.0]
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 177MB/s]
calibrating ...
  t=  1000000       no-ADC uniform      WN=on  top1= 32.2
  t=  1000000       no-ADC uniform      WN=off top1= 70.6
  t=  1000000       no-ADC layer_aware  WN=on  top1= 63.8
  t=  1000000       no-ADC layer_aware  WN=off top1= 69.8
  t=  1000000    ADC-clamp uniform      WN=on  top1= 65.8
  t=  1000000    ADC-clamp uniform      WN=off top1= 67.6
  t=  1000000    ADC-clamp layer_aware  WN=on  top1= 66.0
  t=  1000000    ADC-clamp layer_aware  WN=off top1= 66.4
  t=  1000000  ADC-noclamp uniform      WN=on  top1= 65.8
  t=  1000000  ADC-noclamp uniform      WN=off top1= 67.6
  t=  1000000  ADC-noclamp layer_aware  WN=on  top1= 66.0
  t=  1000000  ADC-noclamp layer_aware  WN=off top1= 66.4

=== decomposition, t=1000000, 5-bit ADC, 500 imgs ===
     

In [3]:
!ls /content/imagenet_val | head    # 应当看到一堆 n0xxxxxxx 类别文件夹

ls: cannot access '/content/imagenet_val': No such file or directory


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os, glob

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/PCM_Research7/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

VAL_ROOT = '/content/imagenet_val'
if not (os.path.exists(VAL_ROOT) and len([d for d in os.listdir(VAL_ROOT) if d.startswith('n')]) >= 100):
    !mkdir -p {VAL_ROOT}
    print('Downloading ImageNet val ...')
    !kaggle datasets download -d titericz/imagenet1k-val -p /content --quiet
    print('Unzipping ...')
    !unzip -q /content/imagenet1k-val.zip -d {VAL_ROOT}
    !rm -f /content/imagenet1k-val.zip

cand = glob.glob(f'{VAL_ROOT}/**/n01440764', recursive=True)
VAL_ROOT = os.path.dirname(cand[0]) if cand else VAL_ROOT
print('VAL_ROOT =', VAL_ROOT)

Dataset URL: https://www.kaggle.com/datasets/titericz/imagenet1k-val
License(s): CC0-1.0
Unzipping ...
VAL_ROOT = /content/imagenet_val/imagenet-val


In [5]:
from google.colab import files
files.upload()          # 选 final_imc_sweep.py

Saving final_imc_sweep.py to final_imc_sweep.py


{'final_imc_sweep.py': b'#!/usr/bin/env python3\n"""\nimc_torch.py -- torch (GPU-capable) port of imc_vec.py\'s INT8 hybrid SRAM/PCM-IMC\nforward for LeNet. Same math as the numpy reference (imc_vec.py); runs on cuda for\nspeed on Colab. Validated to reproduce imc_vec predictions image-for-image, and to\nmatch the RTL golden bit-exactly at the no-drift/no-ADC/no-noise point.\n\ndtype: float64 reproduces numpy exactly; float32 is fast on GPU and is exact for\nLeNet layer sizes (all integer accumulations stay below 2^24). For deeper nets\n(ImageNet) use float64 or int accumulation.\n"""\n\nimport math\nimport torch\nimport numpy as np\ntry:\n    from imc_vec import load_model, read_hex_bytes, LAYERS, WCOUNT  # LeNet-only; not used by ResNet\nexcept Exception:\n    load_model = read_hex_bytes = LAYERS = WCOUNT = None  # ResNet path uses imc_matmul_t directly\n\nDEFAULT_CFG = {\n    "cellBit": 2, "n_mag_cells": 4, "subArray": 64, "in_bits": 8,\n    "adc_bits": None, "drift_t": 1.0, "comp":

In [6]:
import shutil
shutil.copy('/content/final_imc_sweep.py',
            '/content/drive/MyDrive/PCM_Research7/final_imc_sweep.py')
print('saved to Drive')

saved to Drive


In [7]:
%cd /content/drive/MyDrive/PCM_Research7
!python3 final_imc_sweep.py --val_root {VAL_ROOT}

/content/drive/MyDrive/PCM_Research7
subset 2000  device cuda  budget 0.5  seeds 3  adc 5-bit  drift [1.0, 100.0, 10000.0, 1000000.0]  readouts ['noadc', 'adc_pre']
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:03<00:00, 27.2MB/s]
calibrating activation scales ...
[anchor] clean INT8 top1=76.4  (main-run reference 76.4; should match closely)
plan 72 cells | 0 done | 72 to run
[1/72] noadc    t=        1 none         seed=0 top1=  5.5 (3s)  ETA ~2 min + next readout group
[2/72] noadc    t=        1 none         seed=1 top1=  7.9 (3s)  ETA ~2 min + next readout group
[3/72] noadc    t=        1 none         seed=2 top1=  4.9 (3s)  ETA ~2 min + next readout group
[4/72] noadc    t=        1 uniform      seed=0 top1= 55.1 (3s)  ETA ~2 min + next readout group
[5/72] noadc    t=        1 uniform      seed=1 top1= 55.2 (3s)  ETA ~2 min + next readout group
[6/72] noadc    t=        1 